# CS2309 — SwiftEdit WebUI T4 (`fp16_disk_xformers`)

Demo Gradio trên **Google Colab T4**:

| Thành phần | Chi tiết |
|------------|----------|
| Script | [`scripts/app_gradio_t4_xformers.py`](../scripts/app_gradio_t4_xformers.py) |
| Config | **`fp16_disk_xformers`** = FP16 disk + xFormers MEA + EditCache |
| Weights | **Drive trước** (`swiftedit_weights_fp16`); thiếu thì tải + **lưu lại Drive** |
| UI | Tab edit prompt · Tab xóa vật thể (khoanh vùng) |

### Khác notebook WebUI cũ

| | `CS2309_SwiftEdit_webui.ipynb` | Notebook này |
|--|--------------------------------|--------------|
| Máy | Mac MPS hoặc Colab | **Chỉ Colab T4 (CUDA)** |
| Weights | FP32 trên disk + compute fp16 | **FP16 trên disk** |
| xFormers | Không | **Có** |
| Drive | Không bắt buộc | **Ưu tiên** `/content/drive/MyDrive/CS2309/swiftedit_weights_fp16` |

### Cách chạy (Colab)

1. Runtime → **T4 GPU**
2. Cell **①** clone repo + kiểm tra GPU
3. Cell **②** pip (gradio, xformers) — không tải weights ở đây
4. Cell **③** launch WebUI (`--share` → link `*.gradio.live`)

Repo private: Colab Secrets → `GITHUB_TOKEN`.

> Mặc định `SAVE_TO_DRIVE=True`: lần đầu tải/convert xong sẽ copy lên Drive; lần sau chỉ symlink.
>
> Hoặc upload sẵn `swiftedit_weights_fp16` từ Mac để bỏ Qualcomm ~10GB.


### ① Clone repo + kiểm tra GPU T4

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

_COLAB_GPU_ERR = (
    "Colab chưa có GPU CUDA (cần T4).\n"
    "• Colab web: Runtime → Change runtime type → T4 GPU\n"
    "• Extension: New Colab Server → GPU → T4, rồi Restart kernel"
)


def _check_colab_gpu() -> None:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not (r.stdout or "").strip():
        raise RuntimeError(_COLAB_GPU_ERR)
    names = [ln.strip() for ln in r.stdout.strip().splitlines() if ln.strip()]
    print("GPU OK:", ", ".join(names))


REPO_SLUG = "NguyenKz/CS2309.CH201"
USE_PRIVATE_REPO = True


def _colab_repo_url():
    public_url = f"https://github.com/{REPO_SLUG}.git"
    if not USE_PRIVATE_REPO:
        return public_url
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        if not token:
            raise ValueError("Thiếu GITHUB_TOKEN")
        return f"https://{token}@github.com/{REPO_SLUG}.git"
    except Exception as e:
        print(f"Không lấy GITHUB_TOKEN ({e}) — fallback repo public.")
        return public_url


COLAB_REPO_DIR = Path("/content/CS2309.CH201")

if not IN_COLAB:
    raise RuntimeError(
        "Notebook này dành cho Google Colab T4.\n"
        "Local Mac: dùng notebooks/CS2309_SwiftEdit_webui.ipynb + scripts/app_gradio.py"
    )

_check_colab_gpu()
REPO_URL = _colab_repo_url()

if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
    print(f"Cloning github.com/{REPO_SLUG} → {COLAB_REPO_DIR} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)],
        check=True,
    )
else:
    print("Repo đã có:", COLAB_REPO_DIR)

PROJECT_ROOT = COLAB_REPO_DIR
os.environ.setdefault("HF_HOME", "/content/huggingface")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")

APP_SCRIPT = PROJECT_ROOT / "scripts" / "app_gradio_t4_xformers.py"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("app_gradio_t4_xformers.py:", APP_SCRIPT.is_file())
if not APP_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Thiếu {APP_SCRIPT} — pull/push nhánh có scripts/app_gradio_t4_xformers.py"
    )

### ② Setup pip (gradio + xformers)

Weights **không** tải ở cell này — script app sẽ:
1. Mount Drive (nếu path trỏ `/content/drive/...`)
2. Symlink `swiftedit_weights_fp16` nếu có trên Drive
3. Thiếu → tải Qualcomm + convert fp16 (lâu / tốn disk)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "gradio>=5,<6",
        "huggingface-hub<1.0",
        "xformers",
    ],
    check=True,
)

# Deps SwiftEdit (nhẹ — setup_colab đầy đủ nếu thiếu torch/diffusers)
req = PROJECT_ROOT / "SwiftEdit" / "requirements.txt"
if req.is_file():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True,
    )

import gradio as gr
import torch
import xformers

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("xformers:", getattr(xformers, "__version__", "?"))
print("gradio:", gr.__version__)
print("Setup OK — chạy cell ③ Launch.")

### ③ Launch WebUI (`fp16_disk_xformers`)

- Đợi link **`*.gradio.live`** (tự `--share` trên Colab)
- Lần đầu: mount Drive + (nếu thiếu) tải/convert + **lưu Drive** + nạp model — có thể lâu
- Dừng: **Interrupt kernel** (■)

**Cấu hình Drive (đổi nếu bạn upload khác chỗ):**


In [ ]:
import socket

# --- Đường dẫn Drive (đổi nếu cần) ---
DRIVE_FP16 = "/content/drive/MyDrive/CS2309/swiftedit_weights_fp16"
DRIVE_FP32 = "/content/drive/MyDrive/CS2309/swiftedit_weights"

# True = thiếu Drive → tải Qualcomm (~10GB+). False = chỉ dùng Drive/local.
ALLOW_DOWNLOAD = True
# True = sau tải/convert → copy lên Drive (lần sau khỏi tải lại)
SAVE_TO_DRIVE = True

PORT_START = 7860


def _free_port(start: int = 7860, count: int = 20) -> int:
    for port in range(start, start + count):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                s.bind(("0.0.0.0", port))
                return port
            except OSError:
                continue
    return start


def _gpu_used_ratio() -> float | None:
    r = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not r.stdout.strip():
        return None
    used, total = (float(x.strip()) for x in r.stdout.strip().split(","))
    print(f"GPU VRAM: {used:.0f} / {total:.0f} MiB đang dùng")
    return used / total if total else None


ratio = _gpu_used_ratio()
if ratio is not None and ratio > 0.85:
    raise RuntimeError(
        "GPU gần đầy — Restart kernel rồi chạy lại ①②③.\n"
        "Tránh nạp model hai lần trong cùng kernel."
    )

PORT = _free_port(PORT_START)
cmd = [
    sys.executable,
    "-u",
    str(APP_SCRIPT),
    "--drive-fp16",
    DRIVE_FP16,
    "--drive-fp32",
    DRIVE_FP32,
    "--port",
    str(PORT),
    "--share",
]
if not ALLOW_DOWNLOAD:
    cmd.append("--no-download")
if SAVE_TO_DRIVE:
    cmd.append("--save-to-drive")
else:
    cmd.append("--no-save-to-drive")

run_env = os.environ.copy()
run_env["PYTHONUNBUFFERED"] = "1"

print("Config: fp16_disk_xformers + EditCache")
print("SAVE_TO_DRIVE:", SAVE_TO_DRIVE, "| ALLOW_DOWNLOAD:", ALLOW_DOWNLOAD)
print("Lệnh:", " ".join(cmd))
print("\n" + "=" * 60)
print("Colab: đợi link *.gradio.live (sau khi load model).")
print("Tab 1: Chỉnh sửa bằng prompt  |  Tab 2: Xóa vật thể")
print("Dừng: Interrupt kernel (■)")
print("=" * 60 + "\n")

subprocess.run(cmd, cwd=PROJECT_ROOT, check=True, env=run_env)
